In [42]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import os

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

appointments = pd.read_sql("SELECT * FROM appointments", engine)
providers = pd.read_sql("SELECT provider_id, specialty, primary_location_id FROM providers", engine)

print(appointments.shape)
appointments.head()

(199420, 12)


,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2022-07-20,2022-07-20,P2,L1,PT1,PY1,Follow-up,False,Completed,125.50,0.81
1,A2,2022-07-20,2022-07-19,P2,L1,PT1,PY1,Follow-up,False,Completed,75.40,0.83
2,A3,2022-07-20,2022-07-02,P2,L1,PT2,PY3,New Patient Consult,True,Completed,258.34,2.47
3,A4,2022-07-20,2022-07-03,P2,L1,PT2,PY3,Follow-up,False,Completed,163.69,1.15
4,A5,2022-07-20,2022-07-06,P2,L1,PT3,PY5,New Patient Consult,True,Completed,297.58,1.97


In [17]:
df = appointments[appointments['status'].isin(['Completed', 'No-Show'])].copy()
df['no_show'] = (df['status'] == 'No-Show').astype(int)
print(df['no_show'].value_counts())
print(df['no_show'].mean())  # what % are no-shows overall

no_show
0    165110
1     24168
Name: count, dtype: int64
0.1276852037743425


In [18]:
df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu,no_show
0,A1,2022-07-20,2022-07-20,P2,L1,PT1,PY1,Follow-up,False,Completed,125.50,0.81,0
1,A2,2022-07-20,2022-07-19,P2,L1,PT1,PY1,Follow-up,False,Completed,75.40,0.83,0
2,A3,2022-07-20,2022-07-02,P2,L1,PT2,PY3,New Patient Consult,True,Completed,258.34,2.47,0
3,A4,2022-07-20,2022-07-03,P2,L1,PT2,PY3,Follow-up,False,Completed,163.69,1.15,0
4,A5,2022-07-20,2022-07-06,P2,L1,PT3,PY5,New Patient Consult,True,Completed,297.58,1.97,0


In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 189278 entries, 0 to 199419
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   appointment_id    189278 non-null  str    
 1   date              189278 non-null  object 
 2   booked_date       188890 non-null  object 
 3   provider_id       189278 non-null  str    
 4   location_id       189278 non-null  str    
 5   patient_id        189278 non-null  str    
 6   payer_id          189278 non-null  str    
 7   appointment_type  189278 non-null  str    
 8   is_new_patient    189278 non-null  bool   
 9   status            189278 non-null  str    
 10  revenue           187629 non-null  float64
 11  rvu               189278 non-null  float64
 12  no_show           189278 non-null  int64  
dtypes: bool(1), float64(2), int64(1), object(2), str(7)
memory usage: 19.0+ MB


In [20]:
# Convert dates
df['date'] = pd.to_datetime(df['date'], errors='coerce', format='%Y-%m-%d')
df['booked_date'] = pd.to_datetime(df['booked_date'], errors='coerce', format='%Y-%m-%d')

# Likely strongest predictor, days ahead of when booked
df['lead_time_days'] = (df['date'] - df['booked_date']).dt.days

# Null times do not contribute
print(f"Null lead times: {df['lead_time_days'].isna().sum()}")

# 0 = Monday, Sundays are always closed
df['day_of_week'] = df['date'].dt.day_of_week
print(df.groupby('day_of_week')['no_show'].mean())

# For seasonality perhaps
df['month'] = df['date'].dt.month

# Merge provider specialty and location
df = df.merge(providers, on='provider_id', how='left')

Null lead times: 388
day_of_week
0    0.125997
1    0.127713
2    0.129168
3    0.129699
4    0.125529
5    0.128380
Name: no_show, dtype: float64


In [28]:
# Does lead time actually correlate with no-show rate?
# Too little data for 30-999?
df.groupby(pd.cut(df['lead_time_days'], bins=[0,3,7,14,30,999]))['no_show'].mean()

lead_time_days
(0, 3]       0.115492
(3, 7]       0.118687
(7, 14]      0.133906
(14, 30]     0.165993
(30, 999]    0.000000
Name: no_show, dtype: float64

In [22]:
# Value verification
print(df['lead_time_days'].value_counts().sort_index().tail(30))
print(df[df['lead_time_days'].between(14, 30)].shape)

lead_time_days
2.0      9025
3.0     10532
4.0     12075
5.0     12950
6.0     13783
7.0     13885
8.0     13666
9.0     13075
10.0    11821
11.0    10407
12.0     9058
13.0     7627
14.0     6135
15.0     4890
16.0     3664
17.0     2666
18.0     1938
19.0     1329
20.0      884
21.0      615
22.0      349
23.0      229
24.0      109
25.0       75
26.0       29
27.0       26
28.0       11
29.0        4
30.0        2
31.0        1
Name: count, dtype: int64
(22955, 18)


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 189278 entries, 0 to 189277
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype        
---  ------               --------------   -----        
 0   appointment_id       189278 non-null  str          
 1   date                 189278 non-null  datetime64[s]
 2   booked_date          188890 non-null  datetime64[s]
 3   provider_id          189278 non-null  str          
 4   location_id          189278 non-null  str          
 5   patient_id           189278 non-null  str          
 6   payer_id             189278 non-null  str          
 7   appointment_type     189278 non-null  str          
 8   is_new_patient       189278 non-null  bool         
 9   status               189278 non-null  str          
 10  revenue              187629 non-null  float64      
 11  rvu                  189278 non-null  float64      
 12  no_show              189278 non-null  int64        
 13  lead_time_days       188890 non-null  fl

In [29]:
num_features = ['lead_time_days', 'day_of_week', 'month', 'is_new_patient']
cat_features = ['appointment_type', 'specialty', 'location_id', 'payer_id', 'provider_id']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

In [ ]:
# Make df
feature_cols = num_features + cat_features

X = df[feature_cols].copy()
y = df['no_show']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train no-show rate: {y_train.mean():.3f}")
print(f"Test no-show rate:  {y_test.mean():.3f}")

Train: (151422, 9), Test: (37856, 9)
Train no-show rate: 0.128
Test no-show rate:  0.128


In [41]:
# Train and predict
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Encoded results
encoded_cols = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder'].get_feature_names_out()
print(encoded_cols)

['x0_Follow-up' 'x0_Injection/Procedure' 'x0_New Patient Consult'
 'x0_Physical Therapy' 'x0_Post-Op Check' 'x1_Hand & Upper Extremity'
 'x1_Orthopedic Surgery' 'x1_Pain Management'
 'x1_Physical Medicine & Rehab' 'x1_Physical Therapy' 'x1_Spine Surgery'
 'x1_Sports Medicine' 'x1_Unknown' 'x2_L1' 'x2_L2' 'x2_L3' 'x2_L4' 'x2_L5'
 'x2_L6' 'x2_L7' 'x3_PY1' 'x3_PY2' 'x3_PY3' 'x3_PY4' 'x3_PY5' 'x3_PY6'
 'x3_PY7' 'x4_P1' 'x4_P10' 'x4_P11' 'x4_P12' 'x4_P13' 'x4_P14' 'x4_P15'
 'x4_P16' 'x4_P17' 'x4_P18' 'x4_P19' 'x4_P2' 'x4_P20' 'x4_P21' 'x4_P22'
 'x4_P23' 'x4_P24' 'x4_P25' 'x4_P26' 'x4_P27' 'x4_P3' 'x4_P4' 'x4_P5'
 'x4_P6' 'x4_P7' 'x4_P8' 'x4_P9' 'x4_UNK']


In [ ]:
# Evaluate
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred, target_names=['Completed', 'No-Show']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.6421883975144564
              precision    recall  f1-score   support

   Completed       0.91      0.58      0.71     33022
     No-Show       0.18      0.62      0.28      4834

    accuracy                           0.59     37856
   macro avg       0.55      0.60      0.49     37856
weighted avg       0.82      0.59      0.66     37856

Confusion Matrix:
[[19222 13800]
 [ 1816  3018]]
